# sweep-config-dict — worked example 3: Sweep Config — bayes search, two log-uniform params

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sweep-config-dict`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Bayesian search (`'method': 'bayes'`) uses previous run results to choose the next set of hyperparameters. It requires the `'metric'` block to know which direction is better. For parameters whose optimal values span many orders of magnitude — such as learning rate and weight decay — `'log_uniform_values'` is the correct distribution because it samples uniformly on the logarithmic scale.

## Worked solution

**Step 1 — Require the metric block for bayes.**
Unlike random or grid, bayes MUST have a metric block. Without it, wandb will refuse to create the sweep.

**Step 2 — Set up log-uniform parameters.**
`'distribution': 'log_uniform_values'` with `'min'` and `'max'` specifies the endpoints in value space (not log space). wandb internally takes `log(min)` and `log(max)` for sampling. So `min=1e-5, max=1e-1` means any value between 0.00001 and 0.1 can be sampled with equal probability per log-unit.

**Step 3 — Add a fixed parameter.**
Passing `'value': 'adam'` (singular, not `'values'`) fixes a hyperparameter and tells wandb not to vary it. This is useful when you want to document a constant in the config without running multiple values.

**Step 4 — Assemble and verify.**
Check that the three top-level keys are present and that each parameter spec has exactly one of `value`, `values`, or `distribution`.

In [ ]:
def build_bayes_sweep_config(metric_name: str) -> dict:
    return {
        'method': 'bayes',
        'metric': {
            'name': metric_name,
            'goal': 'minimize',
        },
        'parameters': {
            'lr': {
                'distribution': 'log_uniform_values',
                'min': 1e-5,
                'max': 5e-2,
            },
            'weight_decay': {
                'distribution': 'log_uniform_values',
                'min': 1e-6,
                'max': 1e-3,
            },
            'optimizer': {
                'value': 'adam',
            },
        },
    }

# Demonstrate
cfg = build_bayes_sweep_config('val_loss')
print('method:', cfg['method'])                # bayes
print('metric:', cfg['metric'])                # {'name': 'val_loss', 'goal': 'minimize'}
print('lr spec:', cfg['parameters']['lr'])     # log_uniform_values
print('opt spec:', cfg['parameters']['optimizer'])  # fixed